In [6]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
import os
import sys
import time

# fix an issue with the method of running in the notebook
# I probably just need to look into python packaging more
sys.path.append(os.path.join(os.getcwd(), "mlpng"))

import numpy as np

import tensorflow as tf
from tensorflow.keras import Input
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

from mlpng.utils import Config
from mlpng.utils.tf.dataloaders import AlmLoader
from mlpng.utils.tf.plots import plot_histogram, plot_metrics, plot_predictions
from mlpng.utils.tf.callbacks import TimedLoggingCallback, WarmupLearningRate

from mlpng.attn_alm import alm_model

In [8]:
s = Config("settings/ul_nn_128_large.json", False)

MAX_EPOCHS = 300
BATCH_SIZE = 32

# just some info for the model name
timestamp = int(time.time())
model_settings = {
    "dropout_rate": 0.3,
    "name": f"tester",
}

data_loader_args = {
    "shuffle": True,
    "seed": None,
    "batch_size": BATCH_SIZE,
    "cache": True,
    "shuffle_buffer": 1000,
}

# additional metrics we are intrested in
metrics = ["mean_absolute_error"]

In [9]:
lr_schedule = WarmupLearningRate(
    warmup_learning_rate=1e-8,  # start small
    warmup_steps=1e5,
    warmup_scale=50,
    warmup_scale_steps=1,
    warmed_learning_rate=1e-3,
    decay_steps=1000,
    decay_rate=0.95,
    staircase=True,
)

WarmupLearningRate: Warmup Range: 1e-08 -> 0.050000010000000004


In [10]:
strategy = tf.distribute.MirroredStrategy()
print(strategy.num_replicas_in_sync)
# strategy = tf.distribute.OneDeviceStrategy(device="/CPU:0")
num_gpus = strategy.num_replicas_in_sync

data_loader = AlmLoader(
    s.alm_file_complete, num_replicas=num_gpus, **data_loader_args
)
train_dataset, test_dataset, val_dataset = data_loader.get_split(0.8, 0.1, 0.1)

with strategy.scope():
    opt = Adam(learning_rate=lr_schedule)

    model = alm_model(Input(data_loader.shape), **model_settings)
    # model = simple_transformer(Input(data_loader.shape), **model_settings)

    model.compile(optimizer=opt, loss=tf.keras.losses.mse, metrics=metrics)

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1', '/job:localhost/replica:0/task:0/device:GPU:2', '/job:localhost/replica:0/task:0/device:GPU:3', '/job:localhost/replica:0/task:0/device:GPU:4', '/job:localhost/replica:0/task:0/device:GPU:5', '/job:localhost/replica:0/task:0/device:GPU:6', '/job:localhost/replica:0/task:0/device:GPU:7')
22-Feb-24 18:45:50 - tensorflow - INFO - Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1', '/job:localhost/replica:0/task:0/device:GPU:2', '/job:localhost/replica:0/task:0/device:GPU:3', '/job:localhost/replica:0/task:0/device:GPU:4', '/job:localhost/replica:0/task:0/device:GPU:5', '/job:localhost/replica:0/task:0/device:GPU:6', '/job:localhost/replica:0/task:0/device:GPU:7')


8


In [11]:
model.summary()

Model: "tester"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 2, 500, 500)]        0         []                            
                                                                                                  
 dense (Dense)               (None, 2, 500, 512)          256512    ['input_1[0][0]']             
                                                                                                  
 dense_1 (Dense)             (None, 2, 500, 128)          65664     ['dense[0][0]']               
                                                                                                  
 dense_2 (Dense)             (None, 2, 500, 1)            129       ['dense_1[0][0]']             
                                                                                             

In [12]:
callbacks = [
    # We use earlystoping to prevent overfitting
    EarlyStopping(
        monitor="val_loss",
        patience=20,
        verbose=1,
        restore_best_weights=True,
        start_from_epoch=50,
    ),
    # TimedLoggingCallback(print_frequency=60),
    # TensorBoard(
    #     log_dir=f"{s.tb_dir}/{model_settings['name']}",
    #     histogram_freq=1,
    # ),
]

In [13]:
if False:
    import wandb
    from wandb.keras import WandbMetricsLogger

    # wandb.tensorboard.patch(root_logdir=s.tb_dir)

    wandb.init(
        project="mlpng",
        tags=["tester", "dev"],
        config=s.settings | model_settings,
        dir="data",
        sync_tensorboard=True,
    )

    # Add the wandb logger to the callbacks, so it is used
    callbacks.append(WandbMetricsLogger())

wandb: Currently logged in as: jbrandons (mlpng). Use `wandb login --relogin` to force relogin


In [14]:
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=MAX_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

2024-02-22 18:46:02.137120: W tensorflow/core/grappler/optimizers/data/auto_shard.cc:553] The `assert_cardinality` transformation is currently not handled by the auto-shard rewrite and will be removed.


Epoch 1/300
INFO:tensorflow:Collective all_reduce tensors: 20 all_reduces, num_devices = 8, group_size = 8, implementation = CommunicationImplementation.NCCL, num_packs = 1
22-Feb-24 18:46:03 - tensorflow - INFO - Collective all_reduce tensors: 20 all_reduces, num_devices = 8, group_size = 8, implementation = CommunicationImplementation.NCCL, num_packs = 1
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
22-Feb-24 18:46:07 - tensorflow - INFO - Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
22-Feb-24 18:46:07 - tensorflow - INFO - Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:

2024-02-22 18:46:21.592302: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:422] ShuffleDatasetV3:25: Filling up shuffle buffer (this may take a while): 617 of 1000
2024-02-22 18:46:27.394903: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:452] Shuffle buffer filled.
2024-02-22 18:46:46.148346: I external/local_xla/xla/service/service.cc:168] XLA service 0x154df5116590 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2024-02-22 18:46:46.148382: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA A100-SXM4-80GB, Compute Capability 8.0
2024-02-22 18:46:46.148389: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (1): NVIDIA A100-SXM4-80GB, Compute Capability 8.0
2024-02-22 18:46:46.148396: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (2): NVIDIA A100-SXM4-80GB, Compute Capability 8.0
2024-02-22 18:46:46.148402: I external/local_xla/xla/service/service.cc:1

179/312 [================>.............] - ETA: 7:46 - loss: 334589.5938 - mean_absolute_error: 501.2271

KeyboardInterrupt: 

In [ ]:
# Lets plot the predictions from the unseen test set
y_pred = model.predict(test_dataset, verbose=0).flatten()
y_test = np.concatenate([y.numpy() for _, y in test_dataset])

In [ ]:
# Plot the loss curves and metrics
plot_metrics(history, metrics=["loss"] + metrics)
plot_predictions(y_test, y_pred)
plot_histogram(y_test, y_pred)